# ScortexIA/laurelia — TPU Training (flujo constante)

Entrena transformer (GQA+RoPE+SwiGLU) y sube a HF cada 10min con tag `gens0x`.

**Setup:**
1. Colab Secrets → `MY_KEY` = token HF (write)
2. Entorno → TPU v2-8

**Pipeline:**
1. Descarga primeros 60MB de Wikipedia → entrena BPE tokenizer → sube a HF
2. Entrena modelo en streaming
3. Push cada 10 min a `ScortexIA/laurelia` @ `gens0x`

In [ ]:
# ── 1. Instalar ──
print('Installing...')
!pip install -q torch torchvision torch_xla[tpu] libtpu \
    safetensors tokenizers requests huggingface_hub \
    pyarrow datasets
print('Done.')

In [ ]:
# ── 2. Clonar repo (opcional, para pesos Rust) ──
import os
REPO = 'https://github.com/emanuelbertey/Evil-Inference-Code'
BRANCH = 'llamaburn'
DIR = '/content/Evil-Inference-Code'
if not os.path.exists(DIR):
    !git clone --branch {BRANCH} {REPO} {DIR}
%cd {DIR}

In [ ]:
# ── 3. Token HF ──
from google.colab import userdata
MY_KEY = userdata.get('MY_KEY') or ''
if MY_KEY:
    os.environ['MY_KEY'] = MY_KEY
    print('HF token loaded.')
else:
    print('WARNING: sin MY_KEY no se sube a HF.')

In [ ]:
# ── 4. Crear python_rust/ en Colab ──
PKG = '/content/python_rust'
os.makedirs(PKG, exist_ok=True)

# --- __init__.py ---
init_py = '''
from .norm import RMSNorm
from .rope import apply_rope, apply_rope_partial
from .attention import repeat_kv, GQAAttention
from .ffn import SwiGLUFFN
from .block import TransformerBlock
from .model import TransformerLM
from .tokenizer import BPEWrapper
CONFIG = {
    'vocab_size': 16000, 'd_model': 256, 'num_layers': 6,
    'num_heads': 8, 'num_kv_groups': 4, 'max_seq_len': 128,
    'batch_size': 16, 'stride': 128, 'grad_accum': 2,
    'lr': 3e-4, 'warmup_steps': 50, 'total_steps': 100000,
    'lr_min_ratio': 0.2, 'norm_eps': 1e-5, 'ffn_expansion': 4.0,
    'ffn_round_to': 64, 'attn_dropout': 0.0, 'ffn_dropout': 0.0,
    'residual_dropout': 0.0, 'max_norm': 1.0,
    'hf_repo': 'ScortexIA/laurelia', 'hf_tag': 'gens0x',
    'push_every_minutes': 10,
}
'''
with open(f'{PKG}/__init__.py', 'w') as f: f.write(init_py)

# --- norm.py ---
with open(f'{PKG}/norm.py', 'w') as f: f.write('''
import torch.nn as nn
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        denom = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / denom * self.weight
''')

# --- rope.py ---
with open(f'{PKG}/rope.py', 'w') as f: f.write('''
import torch
def _trig(q, k, offset, rd=None):
    B,S,H,D=q.shape; hh=(rd or D)//2
    inv=torch.tensor([1.0/(10000.0**(2*i/D)) for i in range(hh)],
                     dtype=torch.float32,device=q.device)
    pos=torch.arange(offset,offset+S,dtype=torch.float32,device=q.device)
    fr=torch.outer(pos,inv)
    return fr.cos()[None,:,None,:], fr.sin()[None,:,None,:], hh
def _rot(x,cos,sin,hh):
    a,b=x[...,:hh],x[...,hh:2*hh]
    return torch.cat([a*cos-b*sin,a*sin+b*cos],-1)
def apply_rope(q,k,offset=0):
    cos,sin,hh=_trig(q,k,offset)
    return _rot(q,cos,sin,hh), _rot(k,cos,sin,hh)
def apply_rope_partial(q,k,offset=0,rp=0.5):
    D=q.shape[-1]; rd=int(D*rp)//2*2
    if rd>=D or rd==0: return apply_rope(q,k,offset)if rd>=D else(q,k)
    cos,sin,hh=_trig(q,k,offset,rd)
    qo=torch.cat([_rot(q[...,:rd],cos,sin,hh),q[...,rd:]],-1)
    ko=torch.cat([_rot(k[...,:rd],cos,sin,hh),k[...,rd:]],-1)
    return qo,ko
''')

print('python_rust/ created.')

# --- attention.py ---
with open(f'{PKG}/attention.py', 'w') as f: f.write('''
import math,torch,nn,functools as F
from rope import apply_rope
def repeat_kv(x,nh,nkg):
    if nkg==nh: return x
    r=nh//nkg;B,S,G,D=x.shape
    return x[:,:,:,None,:].expand(-1,-1,-1,r,-1).reshape(B,S,nh,D)
class GQAAttention(nn.Module):
    def __init__(s,dm,nh,nkg,hd=None,dp=0.,alc=None,ca=True):
        super().__init__()
        s.nh=nh;s.nkg=nkg;s.hd=hd or dm//nh;s.alc=alc;s.ca=ca
        s.qp=nn.Linear(dm,nh*s.hd,bias=False)
        s.kp=nn.Linear(dm,nkg*s.hd,bias=False)
        s.vp=nn.Linear(dm,nkg*s.hd,bias=False)
        s.op=nn.Linear(nh*s.hd,dm,bias=False)
        s.adp=nn.Dropout(dp)
    def forward(s,x,offset=0,cache=None):
        B,S,D=x.shape
        q=s.qp(x).reshape(B,S,s.nh,s.hd)
        k=s.kp(x).reshape(B,S,s.nkg,s.hd)
        v=s.vp(x).reshape(B,S,s.nkg,s.hd)
        q,k=apply_rope(q,k,offset)
        if cache is not None:
            k=torch.cat([cache['k'],k],1);v=torch.cat([cache['v'],v],1)
        Skv=k.shape[1]
        k=repeat_kv(k,s.nh,s.nkg);v=repeat_kv(v,s.nh,s.nkg)
        q=q.transpose(1,2);k=k.transpose(1,2);v=v.transpose(1,2)
        sc=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(s.hd)
        if s.alc is not None: sc=torch.tanh(sc/s.alc)*s.alc
        if s.ca and S>1:
            m=torch.triu(torch.full((S,Skv),float('-inf'),device=x.device),diagonal=1+Skv-S)
            sc=sc+m.unsqueeze(0).unsqueeze(0)
        aw=F.softmax(sc,-1);aw=s.adp(aw)
        return s.op(torch.matmul(aw,v).transpose(1,2).reshape(B,S,-1))
''')

# --- ffn.py ---
with open(f'{PKG}/ffn.py', 'w') as f: f.write('''
import torch.nn as nn, F
class SwiGLUFFN(nn.Module):
    def __init__(s,dm,di,dp=0.):
        super().__init__()
        s.gup=nn.Linear(dm,2*di,bias=False)
        s.dp2=nn.Linear(di,dm,bias=False)
        s.drop=nn.Dropout(dp)
    def forward(s,x):
        g,u=s.gup(x).chunk(2,-1)
        return s.dp2(s.drop(F.silu(g)*u))
''')

# --- block.py ---
with open(f'{PKG}/block.py', 'w') as f: f.write('''
import torch.nn as nn
from norm import RMSNorm
from attention import GQAAttention
from ffn import SwiGLUFFN
class TransformerBlock(nn.Module):
    def __init__(s,dm,nh,nkg,di,adp=0.,fdp=0.,rdp=0.,alc=None,ca=True,ne=1e-5):
        super().__init__()
        s.an=RMSNorm(dm,eps=ne)
        s.att=GQAAttention(dm,nh,nkg,dropout=adp,attn_logit_cap=alc,causal=ca)
        s.fn=RMSNorm(dm,eps=ne)
        s.ff=SwiGLUFFN(dm,di,dropout=fdp)
        s.rd=nn.Dropout(rdp)
    def forward(s,x,offset=0):
        r=x;h=s.an(x);h=s.att(h,offset);x=r+s.rd(h)
        r=x;h=s.fn(x);h=s.ff(h);return r+s.rd(h)
''')

# --- model.py ---
with open(f'{PKG}/model.py', 'w') as f: f.write('''
import math,torch,nn,F
from block import TransformerBlock
from rope import apply_rope_partial
from attention import repeat_kv
class TransformerLM(nn.Module):
    def __init__(s,vs,dm,nl,nh,nkg,msl=2048,ne=1e-5,fe=4.,fr=64,
                 adp=0.,fdp=0.,rdp=0.,alc=None,ca=True):
        super().__init__()
        di=((int(fe*dm*2./3.)+fr-1)//fr*fr)
        s.emb=nn.Embedding(vs,dm)
        s.lays=nn.ModuleList([TransformerBlock(dm,nh,nkg,di,adp,fdp,rdp,alc,ca,ne)for _ in range(nl)])
        s.fn=nn.LayerNorm(dm,eps=ne,bias=False)
        s.head=nn.Linear(dm,vs,bias=False)
        s.dm=dm;s.nl=nl;s.vs=vs
    def forward(s,ids):
        x=s.emb(ids)
        for l in s.lays: x=l(x,0)
        return s.head(s.fn(x))
    def forward_train_partial_rope(s,ids,rp=0.5):
        x=s.emb(ids)
        for l in s.lays:
            r=x;h=l.an(x);a=l.att;B,S,_=h.shape
            q=a.qp(h).reshape(B,S,a.nh,a.hd)
            k=a.kp(h).reshape(B,S,a.nkg,a.hd)
            v=a.vp(h).reshape(B,S,a.nkg,a.hd)
            q,k=apply_rope_partial(q,k,0,rp)
            k=repeat_kv(k,a.nh,a.nkg);v=repeat_kv(v,a.nh,a.nkg)
            q=q.transpose(1,2);k=k.transpose(1,2);v=v.transpose(1,2)
            sc=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(a.hd)
            if S>1:
                m=torch.triu(torch.full((S,S),float('-inf'),device=x.device),diagonal=1)
                sc=sc+m.unsqueeze(0).unsqueeze(0)
            ao=torch.matmul(F.softmax(sc,-1),v).transpose(1,2).reshape(B,S,-1)
            x=r+a.o_proj(ao);r=x;h=l.fn(x);x=r+l.ff(h)
        return s.head(s.fn(x))
    def save_safetensors(s,path):
        from safetensors.torch import save_file; save_file(s.state_dict(),path)
''')

# --- tokenizer.py ---
with open(f'{PKG}/tokenizer.py', 'w') as f: f.write('''
class BPEWrapper:
    def __init__(s,pt):
        from tokenizers import Tokenizer
        s.tok=Tokenizer.from_file(pt) if isinstance(pt,str) else pt
        s.vs=s.tok.get_vocab_size()
    def encode(s,t): return s.tok.encode(t).ids
    def decode(s,ids): return s.tok.decode(ids,skip_special_tokens=False)
''')

print('All modules written.')

In [ ]:
# ── 5. Entrenar (streaming + push cada 10min) ──
import sys
sys.path.insert(0, '/content')
from python_rust.train import train

train()

---
## Detalles

| Acción | Cada |
|---|---|
| Push a `ScortexIA/laurelia` @ `gens0x` | 10 minutos |
| Checkpoint local | 500 steps |
| Reanuda desde HF si existe | Al iniciar |

Tokenizer: BPE ByteLevel vocab 16000, entrenado en primeros 60MB de Wikipedia.
Modelo: 12.6M params, d_model=256, 6 layers, GQA (8/4), SwiGLU, RoPE.